In [1]:
import pandas as pd
import numpy as np
from inital_clean import (filter_otm_atm, add_columns, cleaning)
from forward_price import (estimate_forward_curve)
from table1 import table1

In [2]:
data = pd.read_csv("dissertation_data.csv")

In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 47374551 entries, 0 to 47374550
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   date             str    
 1   exdate           str    
 2   cp_flag          str    
 3   strike_price     int64  
 4   best_bid         float64
 5   best_offer       float64
 6   volume           int64  
 7   open_interest    int64  
 8   impl_volatility  float64
 9   optionid         int64  
 10  am_settlement    int64  
 11  index_flag       int64  
 12  issuer           str    
 13  exercise_style   str    
dtypes: float64(3), int64(6), str(5)
memory usage: 4.9 GB


In [4]:
data = add_columns(data)
cleaned = cleaning(data)

cleaning summary
initial rows: 47374551
 after IV filter: removed:  4946157 rows
 after maturity filter: removed: 2614909 rows
 after best bid filter : removed 1677566 rows
 after midprice filter: removed 785252
 after strike price filter: removed 9953082 rows
after zero OI/volume filter: removed 567 rows
final test rows: 27397585


In [5]:
cleaned.info()

<class 'pandas.DataFrame'>
Index: 27397018 entries, 0 to 47374547
Data columns (total 13 columns):
 #   Column           Dtype         
---  ------           -----         
 0   date             datetime64[us]
 1   exdate           datetime64[us]
 2   cp_flag          str           
 3   strike_price     float64       
 4   best_bid         float64       
 5   best_offer       float64       
 6   volume           int64         
 7   open_interest    int64         
 8   impl_volatility  float64       
 9   optionid         int64         
 10  am_settlement    int64         
 11  expiry           int64         
 12  mid_price        float64       
dtypes: datetime64[us](2), float64(5), int64(5), str(1)
memory usage: 2.9 GB


In [6]:
cleaned.to_csv("cleaned_research_data.csv", index = False)

explanation: to retain quotes near ATM

In [5]:
forward_data = estimate_forward_curve(cleaned)

In [15]:
forward_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 148194 entries, 0 to 148193
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               148194 non-null  datetime64[us]
 1   exdate             148194 non-null  datetime64[us]
 2   expiry             148194 non-null  int64         
 3   forward_price      148194 non-null  float64       
 4   forward_price_ols  148194 non-null  float64       
 5   discount_factor    148194 non-null  float64       
 6   r_annualized       148194 non-null  float64       
dtypes: datetime64[us](2), float64(4), int64(1)
memory usage: 7.9 MB


In [6]:
df = cleaned.merge(forward_data[["date","exdate","expiry","forward_price"]], on=["date","exdate","expiry"], how="left")
filtered = filter_otm_atm(df)

filter summary
inital rows = 27397018 rows
after filtering call option data = 5147696 rows
after filerting put option data = 12098164 rows
number of dropped rows = 10151158
total kept rows = 17245860 rows


In [7]:
filtered.head()

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,optionid,am_settlement,expiry,mid_price,forward_price
2,2000-01-03,2000-06-17,P,1350.0,40.750,42.750,290,14570,0.257464,10016917,1,165,41.750,1485.288410
3,2000-01-03,2000-06-17,P,1275.0,27.250,29.250,0,5513,0.279201,10024738,1,165,28.250,1485.288410
4,2000-01-03,2000-03-18,C,1575.0,13.500,14.750,43,619,0.184711,10029228,1,74,14.125,1466.459393
5,2000-01-03,2000-02-19,C,1475.0,35.375,37.375,550,3366,0.202513,10032934,1,46,36.375,1461.231821
6,2000-01-03,2000-01-22,C,1535.0,2.000,2.750,77,77,0.176439,10036858,1,18,2.375,1455.623284


In [8]:
table1(filtered)

,Overall,<30,30-90,90-180,180-360,>360
Mean open interest,6.996251e+06,1.762071e+06,2.234126e+06,1.277502e+06,1.145347e+06,736708.716729
Stand.Dev open interest,3.459121e+06,1.070271e+06,1.405998e+06,8.428559e+05,7.437218e+05,446572.594613
CV open interest,4.944249e+01,6.073939e+01,6.293278e+01,6.597688e+01,6.493419e+01,60.617254
Mean trading volume,4.797107e+05,2.233091e+05,1.832474e+05,5.300395e+04,2.910726e+04,11056.898165
Stand.Dev trading volume,2.979121e+05,1.532038e+05,1.350109e+05,4.849680e+04,2.667439e+04,11005.287557
CV trading volume,6.210244e+01,6.860616e+01,7.367683e+01,9.149657e+01,9.164173e+01,99.533227
Mean strike numbers,2.165579e+02,1.470148e+02,1.792091e+02,1.383215e+02,8.284888e+01,78.299129
Stand.Dev strike numbers,1.480547e+02,1.181534e+02,1.370807e+02,1.321427e+02,6.025982e+01,50.987967
CV strike numbers,6.836725e+01,8.036833e+01,7.649203e+01,9.553300e+01,7.273462e+01,65.119455


In [9]:
table = table1(filtered)
table.to_csv("table_results.csv", index = False)

In [10]:
data = filtered.drop(columns=["forward_price"])
data.head()

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,optionid,am_settlement,expiry,mid_price
2,2000-01-03,2000-06-17,P,1350.0,40.750,42.750,290,14570,0.257464,10016917,1,165,41.750
3,2000-01-03,2000-06-17,P,1275.0,27.250,29.250,0,5513,0.279201,10024738,1,165,28.250
4,2000-01-03,2000-03-18,C,1575.0,13.500,14.750,43,619,0.184711,10029228,1,74,14.125
5,2000-01-03,2000-02-19,C,1475.0,35.375,37.375,550,3366,0.202513,10032934,1,46,36.375
6,2000-01-03,2000-01-22,C,1535.0,2.000,2.750,77,77,0.176439,10036858,1,18,2.375


In [14]:
data.head(-10)

,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,optionid,am_settlement,expiry,mid_price
2,2000-01-03,2000-06-17,P,1350.0,40.750,42.750,290,14570,0.257464,10016917,1,165,41.750
3,2000-01-03,2000-06-17,P,1275.0,27.250,29.250,0,5513,0.279201,10024738,1,165,28.250
4,2000-01-03,2000-03-18,C,1575.0,13.500,14.750,43,619,0.184711,10029228,1,74,14.125
5,2000-01-03,2000-02-19,C,1475.0,35.375,37.375,550,3366,0.202513,10032934,1,46,36.375
6,2000-01-03,2000-01-22,C,1535.0,2.000,2.750,77,77,0.176439,10036858,1,18,2.375
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27396983,2025-07-31,2026-06-30,P,6175.0,261.200,262.500,0,167,0.177337,170136845,0,334,261.850
27396984,2025-07-31,2026-06-30,P,6200.0,267.300,268.700,2,1610,0.175716,170136846,0,334,268.000
27396985,2025-07-31,2026-06-30,P,6225.0,273.500,275.100,0,153,0.174091,170136847,0,334,274.300
27396986,2025-07-31,2026-06-30,P,6250.0,279.900,281.500,1,113,0.172438,170136848,0,334,280.700


In [11]:
data.to_csv("cleaned_diss_2000_2025.csv", index = False)

In [17]:
vix = pd.read_csv("vix-2000-2025.csv")

In [18]:
vix.head(-10)

,date,vix
0,2000-01-03,24.21
1,2000-01-04,27.01
2,2000-01-05,26.41
3,2000-01-06,25.73
4,2000-01-07,21.72
...,...,...
6516,2025-10-13,19.03
6517,2025-10-14,20.81
6518,2025-10-15,20.64
6519,2025-10-16,25.31


In [19]:
vix["date"] = pd.to_datetime(vix["date"])

In [20]:
vix.info()

<class 'pandas.DataFrame'>
RangeIndex: 6531 entries, 0 to 6530
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    6531 non-null   datetime64[us]
 1   vix     6527 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 102.2 KB


In [21]:
vix = vix.set_index("date")

In [22]:
vix.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 6531 entries, 2000-01-03 to 2025-10-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   vix     6527 non-null   float64
dtypes: float64(1)
memory usage: 102.0 KB


In [24]:
vix["vix"] = vix["vix"].ffill()  
"""to fill the missing values"""
print(vix["vix"].isna().sum())

0


In [27]:
vix.to_csv("vix-2000-2025-prepped.csv")

note, to load into code:
vix_series = pd.read_csv(
    "vix-2000-2025-prepped.csv",
    index_col = 0.
    parse_dates = True,
)["vix"]